In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

# -- CONFIGURATION -----------------------------------------------------------

countries = ["UK", "US"]
# Example regions: UK counties or US states; replace with your actual regions
regions_by_country = {
    "UK": ["London", "Midlands", "Scotland", "Wales", "Northern Ireland"],
    "US": ["California", "New York", "Texas", "Florida", "Illinois"]
}

# Time range: monthly data
start_date = "2023-01-01"
end_date   = "2024-12-01"  # inclusive; month step

# Output CSV path
output_csv = "tropical_blast_sales_demographics.csv"

# -- HELPER FUNCTIONS --------------------------------------------------------

def generate_monthly_index(start, end):
    return pd.date_range(start=start, end=end, freq="MS")  # Month start

def random_demographics():
    """
    Return a simple demographic snapshot for a region-month.
    Percentages should sum to 1 or close to it.
    Here: age groups and gender split; you may add income, education, etc.
    """
    # Age groups (example splits)
    ages = np.random.dirichlet([2, 3, 2, 1])  # four age groups
    # Gender split
    genders = np.random.dirichlet([5, 5])     # male, female
    
    return {
        "pct_age_18_34": ages[0],
        "pct_age_35_54": ages[1],
        "pct_age_55_plus": ages[2],
        "pct_age_under_18": ages[3],
        "pct_male": genders[0],
        "pct_female": genders[1],
    }

def random_sales_and_financials():
    """
    Generate synthetic sales, cost, and margin info.
    Replace with real values or a direct data source.
    """
    units = int(np.random.normal(loc=5000, scale=1500))
    units = max(units, 200)  # no negative/very tiny
    price_per_unit = np.random.uniform(1.5, 3.5)  # example price in GBP/USD
    revenue = units * price_per_unit
    
    # Cost of goods sold: assume 50–75% of revenue
    cogs_ratio = np.random.uniform(0.50, 0.75)
    cogs = revenue * cogs_ratio
    
    gross_profit = revenue - cogs
    gross_margin = gross_profit / revenue if revenue > 0 else 0.0
    
    # Operating expenses: example 10–20% of revenue
    op_exp_ratio = np.random.uniform(0.10, 0.20)
    operating_expenses = revenue * op_exp_ratio
    
    net_profit = gross_profit - operating_expenses
    net_margin = net_profit / revenue if revenue > 0 else 0.0
    
    return {
        "units_sold": units,
        "revenue": revenue,
        "cogs": cogs,
        "gross_profit": gross_profit,
        "gross_margin": gross_margin,
        "operating_expenses": operating_expenses,
        "net_profit": net_profit,
        "net_margin": net_margin,
    }

# -- MAIN DATA GENERATION ----------------------------------------------------

rows = []
for country in countries:
    for region in regions_by_country[country]:
        for month in generate_monthly_index(start_date, end_date):
            # Synthetic demo data
            demo = random_demographics()
            fin  = random_sales_and_financials()
            
            row = {
                "country": country,
                "region": region,
                "year": month.year,
                "month": month.month,
                # Financials
                "units_sold": fin["units_sold"],
                "revenue": round(fin["revenue"], 2),
                "cogs": round(fin["cogs"], 2),
                "gross_profit": round(fin["gross_profit"], 2),
                "gross_margin_pct": round(fin["gross_margin"] * 100, 2),
                "operating_expenses": round(fin["operating_expenses"], 2),
                "net_profit": round(fin["net_profit"], 2),
                "net_margin_pct": round(fin["net_margin"] * 100, 2),
                # Demographics
                "pct_age_under_18": round(demo["pct_age_under_18"] * 100, 2),
                "pct_age_18_34": round(demo["pct_age_18_34"] * 100, 2),
                "pct_age_35_54": round(demo["pct_age_35_54"] * 100, 2),
                "pct_age_55_plus": round(demo["pct_age_55_plus"] * 100, 2),
                "pct_male": round(demo["pct_male"] * 100, 2),
                "pct_female": round(demo["pct_female"] * 100, 2),
            }
            
            rows.append(row)

df = pd.DataFrame(rows)

# Optionally sort
df = df.sort_values(["country", "region", "year", "month"]).reset_index(drop=True)

# Save to CSV
df.to_csv(output_csv, index=False)

print(f"Generated {len(df)} rows to {output_csv}")

Generated 240 rows to tropical_blast_sales_demographics.csv
